# Stage 2 reranking: validation and test

Owner: Chua Rui Ning, Renee

This notebook runs Stage 2 for LSTM, XGBoost, Gemini, and the fixed 75:25 LSTM:XGBoost RRF on validation, then runs only the selected 75:25 RRF on test (five model/split runs total).

For each configured model/split it saves the base top five and three reranking variants: diversity-only, freshness-only, and freshness-plus-diversity. Validation covers all configured rankers; test reads and reranks only the RRF output. Each run uses only its own split labels and rankings and writes only to its matching folder:

- Evaluation/Stage 2 - Reranking/validation
- Evaluation/Stage 2 - Reranking/test

The article catalogue and diversity constants are shared inputs. Diversity constants are estimated from training data only. Use validation for development; test is for final evaluation after choices are frozen.

In [1]:
%pip -q install numpy pandas pyarrow
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/bt4222_eval

Mounted at /content/drive


## Paths: all validation models, selected test ensemble

Configure full top-200 ranking exports for LSTM, XGBoost, Gemini, and 75:25 RRF on validation, and the 75:25 RRF top-200 export on test. Validation rankings are paired only with validation labels; the test RRF ranking is paired only with test labels. Outputs are split-specific.

In [2]:
from pathlib import Path
import json
import hashlib
import sys
import io
from contextlib import redirect_stdout

import pandas as pd

PROJECT_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder')
EVALUATION_ROOT = PROJECT_ROOT / 'Evaluation'
EVALUATION_CODE_DIR = EVALUATION_ROOT / 'code'
STAGE2_ROOT = EVALUATION_ROOT / 'Stage 2 - Reranking'
SHARED_OUTPUT_DIR = STAGE2_ROOT / 'shared'

TRAIN_LABELS_PATH = PROJECT_ROOT / 'feature_engineering' / 'train_features.parquet'
LABELS_BY_SPLIT = {
    'validation': PROJECT_ROOT / 'feature_engineering' / 'validation_features.parquet',
    'test': PROJECT_ROOT / 'feature_engineering' / 'test_features.parquet',
}
ARTICLES_PATH = EVALUATION_ROOT / 'catalogue' / 'article_union.parquet'
SIMILARITY_CONSTANTS_PATH = (
    SHARED_OUTPUT_DIR / 'train_diversity_constants.json'
)
NAIVE_TIMEZONE = 'UTC'

required_modules = (
    'evaluate.py',
    'rerank.py',
    'similarity.py',
    'compare_models.py',
)
missing_modules = [
    name for name in required_modules
    if not (EVALUATION_CODE_DIR / name).is_file()
]
if missing_modules:
    raise FileNotFoundError(
        f'Missing {missing_modules} under {EVALUATION_CODE_DIR}. '
        'Copy the evaluator Python files into Drive at Evaluation/code/.'
    )
sys.path.insert(0, str(EVALUATION_CODE_DIR))

LSTM_RUN_DIR = (
    PROJECT_ROOT / 'Experiments' / '2.1_lstm' / '2.1.3_improved_run'
    / '20261104_175920_improved_seed42'
)
XGBOOST_RUN_DIR = (
    PROJECT_ROOT / 'Experiments' / '1.3_xgboost' / '20261105_small_seed42'
)
GEMINI_RUN_DIR = (
    PROJECT_ROOT / 'Experiments'
    / '3.1_pretrained_similarity_gemini_individual_history'
    / '20261105_213442_history_seed42'
)
RRF_ROOT = EVALUATION_ROOT / 'ensemble'

MODEL_DEFINITIONS = [
    {
        'name': 'LSTM',
        'rankings_by_split': {
            split: (
                LSTM_RUN_DIR / 'rankings' / split
                / 'submission_top_200.parquet'
            )
            for split in ('validation', 'test')
        },
    },
    {
        'name': 'XGBoost',
        'rankings_by_split': {
            split: (
                XGBOOST_RUN_DIR / 'rankings' / split
                / 'submission_top_200.parquet'
            )
            for split in ('validation', 'test')
        },
    },
    {
        'name': 'Gemini',
        'rankings_by_split': {
            'validation': (
                GEMINI_RUN_DIR / 'rankings' / 'validation'
                / 'submission_top_200.parquet'
            ),
        },
    },
    {
        'name': 'RRF_LSTM75_XGBoost25',
        'rankings_by_split': {
            'validation': (
                RRF_ROOT / 'validation'
                / 'rrf_lstm75_xgboost25_validation_top_200.parquet'
            ),
            'test': (
                RRF_ROOT / 'test'
                / 'rrf_lstm75_xgboost25_test_top_200.parquet'
            ),
        },
    },
]

SPLITS = ('validation', 'test')
OUTPUT_DIRS = {
    split: STAGE2_ROOT / split
    for split in SPLITS
}

A0_HOURS = 0.0
FRESHNESS_HALF_LIFE_HOURS = 24.0
FRESHNESS_WEIGHT = 0.0
CONTINUITY_WEIGHT = 0.0
MMR_LAMBDA = 0.9
OUTPUT_K = 5
EXPORT_K = 200

def model_runs_for_split(split):
    if split == 'validation':
        selected_names = {'LSTM', 'XGBoost', 'Gemini', 'RRF_LSTM75_XGBoost25'}
    elif split == 'test':
        selected_names = {'RRF_LSTM75_XGBoost25'}
    else:
        raise ValueError(f'Unsupported split: {split}')

    return [
        {
            'name': model['name'],
            'rankings': model['rankings_by_split'][split],
        }
        for model in MODEL_DEFINITIONS
        if model['name'] in selected_names
    ]

required_paths = [
    TRAIN_LABELS_PATH,
    ARTICLES_PATH,
    *LABELS_BY_SPLIT.values(),
]
for split in SPLITS:
    required_paths.extend(
        Path(run['rankings'])
        for run in model_runs_for_split(split)
    )
missing = [
    str(path)
    for path in required_paths
    if not Path(path).is_file()
]
if missing:
    raise FileNotFoundError(
        'Required input(s) not found:\n' + '\n'.join(missing)
    )

for split, output_dir in OUTPUT_DIRS.items():
    output_dir.mkdir(parents=True, exist_ok=True)

# Clear cached modules from any earlier notebook/import path.
for module_name in ('evaluate', 'similarity', 'rerank'):
    sys.modules.pop(module_name, None)

from evaluate import (
    catalogue_fingerprints,
    file_provenance,
    load_articles,
    load_rankings,
    main as evaluate_main,
)
from rerank import (
    _request_table,
    _write_table,
    estimate_similarity_constants,
    rerank,
)

print('Drive project:', PROJECT_ROOT)
print('Shared catalogue:', ARTICLES_PATH)
print('Training labels for shared diversity constants:', TRAIN_LABELS_PATH)
for split in SPLITS:
    print(f'{split} labels:', LABELS_BY_SPLIT[split])
    print(f'{split} output:', OUTPUT_DIRS[split])
    for run in model_runs_for_split(split):
        print(f"  {run['name']}: {run['rankings']}")

Drive project: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder
Shared catalogue: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/catalogue/article_union.parquet
Training labels for shared diversity constants: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/feature_engineering/train_features.parquet
validation labels: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/feature_engineering/validation_features.parquet
validation output: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/Stage 2 - Reranking/validation
  LSTM: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/2.1_lstm/2.1.3_improved_run/20261104_175920_improved_seed42/rankings/validation/submission_top_200.parquet
  XGBoost: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/1.3_xgboost/20261105_small_seed42/rankings/validation/submission_top_200.parquet
  Gemini: /c

## Estimate shared diversity fallbacks

The constants are estimated from training requests only and saved under Evaluation/Stage 2 - Reranking/shared. The exact same constants are then used for validation and test, avoiding split-specific tuning of the similarity fallback.

In [3]:
REFIT_SIMILARITY_CONSTANTS = False
SHARED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if REFIT_SIMILARITY_CONSTANTS or not SIMILARITY_CONSTANTS_PATH.is_file():
    constants = estimate_similarity_constants(
        TRAIN_LABELS_PATH,
        ARTICLES_PATH,
        output_path=SIMILARITY_CONSTANTS_PATH,
        split='train',
        sample_pairs=100_000,
        seed=42,
        naive_timezone=NAIVE_TIMEZONE,
    )
else:
    constants = json.loads(
        SIMILARITY_CONSTANTS_PATH.read_text(encoding='utf-8')
    )

print('Shared training-derived constants:', SIMILARITY_CONSTANTS_PATH)
print(json.dumps(constants, indent=2))

Shared training-derived constants: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/Stage 2 - Reranking/shared/train_diversity_constants.json
{
  "version": "shared_article_similarity_v1",
  "s_same": 0.27667393212655605,
  "s_diff": 0.053513104418583785,
  "s_all": 0.08338422006413217,
  "sampling": "uniform request; two distinct eligible articles sampled uniformly without replacement within each pair; repeated request/pair draws discarded",
  "split": "train",
  "seed": 42,
  "sample_pairs_requested": 100000,
  "sample_pairs_used": 100000,
  "pair_counts": {
    "sampled_pairs": 100000,
    "both_have_topics": 98084,
    "topic_pairs_same_category": 13129,
    "topic_pairs_different_category": 84955,
    "topic_pairs_category_missing": 0
  },
  "empty_estimate_fallback": {
    "s_same": null,
    "s_diff": null
  },
  "naive_timezone": "UTC"
}


## Run Stage 2 for validation models and the selected test RRF

For each model/split, always evaluate the base top five. A reranked variant is generated only when it changes the ranking: diversity-only when `MMR_LAMBDA < 1`, freshness-only when `FRESHNESS_WEIGHT > 0`, and freshness-plus-diversity when both conditions hold.

Base is the Stage 1 top-five prefix. Skipped no-op variants are not written or summarized.

Validation runs cover LSTM, XGBoost, Gemini, and RRF; test covers RRF only. Each run is evaluated against matching split labels. Reports, top-five rankings, manifests, and per-request metrics go to the matching split folder. Continuity remains disabled. The fixed settings are exploratory and untuned.

In [4]:
if CONTINUITY_WEIGHT != 0:
    raise ValueError(
        'This controlled comparison fixes continuity off; '
        'set CONTINUITY_WEIGHT = 0.'
    )

articles = load_articles(
    ARTICLES_PATH,
    naive_timezone=NAIVE_TIMEZONE,
)
results_by_split = {split: [] for split in SPLITS}


def evaluate_ranking(
    split,
    labels_path,
    output_dir,
    model_name,
    ranking_path,
    stage_name,
):
    stem = model_name.lower().replace(' ', '_')
    report_path = (
        output_dir
        / f'{stem}_{split}_{stage_name}_report.json'
    )
    per_request_path = (
        output_dir
        / f'{stem}_{split}_{stage_name}_per_request.parquet'
    )
    argv = [
        'evaluate.py',
        '--split', split,
        '--labels', str(labels_path),
        '--articles', str(ARTICLES_PATH),
        '--rankings', str(ranking_path),
        '--similarity-constants', str(SIMILARITY_CONSTANTS_PATH),
        '--export-k', str(OUTPUT_K),
        '--ks', str(OUTPUT_K),
        '--output', str(report_path),
        '--per-request-output', str(per_request_path),
    ]

    old_argv = sys.argv
    try:
        sys.argv = argv
        with redirect_stdout(io.StringIO()):
            evaluate_main()
    finally:
        sys.argv = old_argv

    report = json.loads(report_path.read_text(encoding='utf-8'))
    if report.get('split') != split:
        raise RuntimeError(
            f'{model_name}: evaluator reported split '
            f'{report.get("split")!r}; expected {split!r}'
        )
    results_by_split[split].append({
        'model': model_name,
        'stage': stage_name,
        **report['metrics'],
    })
    return report_path, per_request_path


def save_and_evaluate_rerank(
    split,
    labels_path,
    output_dir,
    requests,
    run,
    base_rankings,
    variant,
    freshness_weight,
    diversity_lambda,
):
    name = run['name']
    ranking_path = Path(run['rankings'])
    stem = name.lower().replace(' ', '_')

    reranked = rerank(
        base_rankings,
        requests,
        articles,
        constants,
        freshness_grace_h=A0_HOURS,
        freshness_half_life_h=FRESHNESS_HALF_LIFE_HOURS,
        freshness_weight=freshness_weight,
        continuity_weight=0.0,
        diversity_lambda=diversity_lambda,
        output_k=OUTPUT_K,
        continuity_scores=None,
    )

    reranked_path = (
        output_dir
        / f'{stem}_{split}_stage2_{variant}_top_{OUTPUT_K}.parquet'
    )
    _write_table(reranked, reranked_path)

    catalogue_id_hash, catalogue_content_hash = catalogue_fingerprints(
        articles
    )
    request_hash = hashlib.sha256(
        '\n'.join(
            sorted(reranked.behaviour_id.astype(str))
        ).encode('utf-8')
    ).hexdigest()
    constants_hash = hashlib.sha256(
        json.dumps(
            constants,
            sort_keys=True,
            separators=(',', ':'),
        ).encode('utf-8')
    ).hexdigest()

    rerank_manifest = {
        'protocol': 'stage2_reranking_ablation_v1',
        'model': name,
        'variant': variant,
        'split': split,
        'request_count': int(len(reranked)),
        'input_export_k': EXPORT_K,
        'output_k': OUTPUT_K,
        'request_id_fingerprint_sha256': request_hash,
        'catalogue_id_fingerprint_sha256': catalogue_id_hash,
        'catalogue_content_fingerprint_sha256': catalogue_content_hash,
        'similarity_constants_sha256': constants_hash,
        'parameters': {
            'a0_h': A0_HOURS,
            'half_life_h': FRESHNESS_HALF_LIFE_HOURS,
            'freshness_weight': freshness_weight,
            'continuity_weight': 0.0,
            'lambda_mmr': diversity_lambda,
        },
        'inputs': {
            'labels': file_provenance(labels_path),
            'articles': file_provenance(ARTICLES_PATH),
            'stage1_rankings': file_provenance(ranking_path),
        },
    }
    manifest_path = reranked_path.with_suffix(
        reranked_path.suffix + '.manifest.json'
    )
    manifest_path.write_text(
        json.dumps(rerank_manifest, indent=2) + '\n',
        encoding='utf-8',
    )

    stage_name = f'stage2_{variant}_top5'
    _, per_request_path = evaluate_ranking(
        split,
        labels_path,
        output_dir,
        name,
        reranked_path,
        stage_name,
    )
    print(f'{split} {name} {variant} ranking -> {reranked_path}')
    print(f'{split} {name} {variant} manifest -> {manifest_path}')
    print(f'{split} {name} {variant} metrics -> {per_request_path}')
    return reranked_path, per_request_path


for split in SPLITS:
    labels_path = LABELS_BY_SPLIT[split]
    output_dir = OUTPUT_DIRS[split]
    requests = _request_table(
        labels_path,
        split,
        NAIVE_TIMEZONE,
    )
    model_runs = model_runs_for_split(split)

    print(
        f'\n===== Stage 2 {split.upper()}: '
        f'{len(model_runs)} models ====='
    )

    # Check no-op behavior independently in each split.
    probe_run = model_runs[0]
    probe_rankings = load_rankings(Path(probe_run['rankings']))
    if probe_rankings.empty:
        raise ValueError(
            f"{split} {probe_run['name']} ranking file is empty"
        )
    probe_id = str(probe_rankings.behaviour_id.iloc[0])
    probe_request = requests.loc[
        requests.behaviour_id.astype(str).eq(probe_id)
    ]
    if len(probe_request) != 1:
        raise ValueError(
            f'No unique {split} request found for no-op check: {probe_id}'
        )
    probe_result = rerank(
        probe_rankings.iloc[[0]],
        probe_request,
        articles,
        constants,
        freshness_grace_h=A0_HOURS,
        freshness_half_life_h=FRESHNESS_HALF_LIFE_HOURS,
        freshness_weight=0.0,
        continuity_weight=0.0,
        diversity_lambda=1.0,
        output_k=OUTPUT_K,
        continuity_scores=None,
    )
    expected_probe = probe_rankings.article_ids.iloc[0][:OUTPUT_K]
    if probe_result.article_ids.iloc[0] != expected_probe:
        raise AssertionError(
            f'{split} no-op reranking failed to preserve the Stage 1 top-five'
        )
    print(f'{split} no-op invariant passed for request {probe_id}.')

    for run in model_runs:
        name = run['name']
        ranking_path = Path(run['rankings'])
        if not ranking_path.is_file():
            raise FileNotFoundError(
                f'{split} {name} ranking not found: {ranking_path}'
            )
        base_rankings = load_rankings(ranking_path)

        # Validate that the Stage 1 input covers exactly this split's requests.
        ranking_ids = set(base_rankings.behaviour_id.astype(str))
        request_ids = set(requests.behaviour_id.astype(str))
        if ranking_ids != request_ids:
            raise ValueError(
                f'{split} {name} request coverage mismatch: '
                f'{len(request_ids - ranking_ids)} missing, '
                f'{len(ranking_ids - request_ids)} unexpected.'
            )

        base_top5 = pd.DataFrame({
            'behaviour_id': base_rankings.behaviour_id.astype(str),
            'article_ids': [
                ids[:OUTPUT_K]
                for ids in base_rankings.article_ids
            ],
        })
        stem = name.lower().replace(' ', '_')
        base_top5_path = (
            output_dir
            / f'{stem}_{split}_stage1_base_top_{OUTPUT_K}.parquet'
        )
        _write_table(base_top5, base_top5_path)
        _, base_per_request = evaluate_ranking(
            split,
            labels_path,
            output_dir,
            name,
            base_top5_path,
            'stage1_base_top5',
        )
        print(
            f'{split} {name} base metrics -> {base_per_request}'
        )

        active_variants = []
        if MMR_LAMBDA < 1.0:
            active_variants.append(('diversity_only', 0.0, MMR_LAMBDA))
        if FRESHNESS_WEIGHT > 0.0:
            active_variants.append(('freshness_only', FRESHNESS_WEIGHT, 1.0))
        if FRESHNESS_WEIGHT > 0.0 and MMR_LAMBDA < 1.0:
            active_variants.append((
                'freshness_diversity',
                FRESHNESS_WEIGHT,
                MMR_LAMBDA,
            ))

        for variant, freshness_weight, diversity_lambda in active_variants:
            save_and_evaluate_rerank(
                split,
                labels_path,
                output_dir,
                requests,
                run,
                base_rankings,
                variant,
                freshness_weight=freshness_weight,
                diversity_lambda=diversity_lambda,
            )

for split in SPLITS:
    print(f'\n{split.upper()} metrics:')
    display(pd.DataFrame(results_by_split[split]))
    print(f'All {split} outputs saved under: {OUTPUT_DIRS[split]}')


===== Stage 2 VALIDATION: 4 models =====
validation no-op invariant passed for request 1026339:221666.
validation LSTM base metrics -> /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/Stage 2 - Reranking/validation/lstm_validation_stage1_base_top5_per_request.parquet
validation LSTM diversity_only ranking -> /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/Stage 2 - Reranking/validation/lstm_validation_stage2_diversity_only_top_5.parquet
validation LSTM diversity_only manifest -> /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/Stage 2 - Reranking/validation/lstm_validation_stage2_diversity_only_top_5.parquet.manifest.json
validation LSTM diversity_only metrics -> /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/Stage 2 - Reranking/validation/lstm_validation_stage2_diversity_only_top5_per_request.parquet
validation XGBoost base metrics -> /content/drive/MyDrive/Colab Noteboo

,model,stage,requests,label_requests_available,requests_without_export,requests_with_known_clicks,requests_without_known_clicks,users,logged_candidate_rows,ranked_catalogue_rows,...,median_article_age_h@5,unknown_age_share@5,age_le_24h_share@5,age_24_72h_share@5,age_gt_72h_share@5,topic_coverage@5,topic_article_coverage@5,intra_list_diversity@5,shared_intra_list_diversity@5,distinct_categories@5
0,LSTM,stage1_base_top5,29110,29110,0,29110,0,1719,344666,145550,...,0.903079,0.0,1.000000,0.000000,0.000000,13.726761,1.000000,0.859451,0.859451,3.153830
1,LSTM,stage2_diversity_only_top5,29110,29110,0,29110,0,1719,344666,145550,...,0.944665,0.0,1.000000,0.000000,0.000000,16.322947,1.000000,0.970042,0.970042,3.737685
2,XGBoost,stage1_base_top5,29110,29110,0,29110,0,1719,344666,145550,...,12.077553,0.0,0.998282,0.000007,0.001711,8.544005,1.000000,0.611808,0.611808,1.734593
3,XGBoost,stage2_diversity_only_top5,29110,29110,0,29110,0,1719,344666,145550,...,12.448401,0.0,0.997066,0.000000,0.002934,11.072655,1.000000,0.813424,0.813424,2.216352
4,Gemini,stage1_base_top5,29110,29110,0,29110,0,1719,344666,145550,...,141.440073,0.0,0.042274,0.212917,0.744809,14.931364,0.999739,0.899874,0.899814,3.390003
5,Gemini,stage2_diversity_only_top5,29110,29110,0,29110,0,1719,344666,145550,...,140.622383,0.0,0.044617,0.213425,0.741958,16.623978,0.999814,0.963070,0.963010,3.697217
6,RRF_LSTM75_XGBoost25,stage1_base_top5,29110,29110,0,29110,0,1719,344666,145550,...,1.620654,0.0,1.000000,0.000000,0.000000,11.734387,1.000000,0.769057,0.769057,2.492099
7,RRF_LSTM75_XGBoost25,stage2_diversity_only_top5,29110,29110,0,29110,0,1719,344666,145550,...,1.469521,0.0,1.000000,0.000000,0.000000,15.275919,1.000000,0.950682,0.950682,3.392271


All validation outputs saved under: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/Stage 2 - Reranking/validation

TEST metrics:


,model,stage,requests,label_requests_available,requests_without_export,requests_with_known_clicks,requests_without_known_clicks,users,logged_candidate_rows,ranked_catalogue_rows,...,median_article_age_h@5,unknown_age_share@5,age_le_24h_share@5,age_24_72h_share@5,age_gt_72h_share@5,topic_coverage@5,topic_article_coverage@5,intra_list_diversity@5,shared_intra_list_diversity@5,distinct_categories@5
0,RRF_LSTM75_XGBoost25,stage1_base_top5,29142,29142,0,29142,0,1757,348951,145710,...,1.667658,0.0,1.0,0.0,0.0,11.886487,1.0,0.780910,0.780910,2.553600
1,RRF_LSTM75_XGBoost25,stage2_diversity_only_top5,29142,29142,0,29142,0,1757,348951,145710,...,1.505711,0.0,1.0,0.0,0.0,15.292773,1.0,0.951782,0.951782,3.383021


All test outputs saved under: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/Stage 2 - Reranking/test


In [5]:
metrics_to_show = [
    'known_click_recall@5',
    'observed_click_ndcg@5',
    'known_click_recall@100',
    'observed_click_ndcg@100',
    'known_click_recall@200',
    'observed_click_ndcg@200',
]

summaries_by_split = {}
for split in SPLITS:
    output_dir = OUTPUT_DIRS[split]
    rows = []

    expected_report_paths = []
    expected_stages = ['stage1_base_top5']
    if MMR_LAMBDA < 1.0:
        expected_stages.append('stage2_diversity_only_top5')
    if FRESHNESS_WEIGHT > 0.0:
        expected_stages.append('stage2_freshness_only_top5')
    if FRESHNESS_WEIGHT > 0.0 and MMR_LAMBDA < 1.0:
        expected_stages.append('stage2_freshness_diversity_top5')
    for run in model_runs_for_split(split):
        stem = run['name'].lower().replace(' ', '_')
        expected_report_paths.extend(
            output_dir / f'{stem}_{split}_{stage}_report.json'
            for stage in expected_stages
        )

    missing_reports = [path for path in expected_report_paths if not path.is_file()]
    if missing_reports:
        raise FileNotFoundError(
            'Expected report(s) missing for this split:\n'
            + '\n'.join(str(path) for path in missing_reports)
        )

    for path in expected_report_paths:
        report = json.loads(path.read_text(encoding='utf-8'))
        if report.get('split') != split:
            raise ValueError(
                f'{path} contains split {report.get("split")!r}; '
                f'expected {split!r}'
            )
        metrics = report.get('metrics', {})
        rows.append({
            'model': report.get('run_config', {}).get('experiment_id')
            or path.stem,
            'stage': report.get('stage', path.stem),
            'split': split,
            'requests': report.get('evaluated_request_count'),
            **{
                metric: metrics.get(metric)
                for metric in metrics_to_show
            },
        })

    if not rows:
        raise FileNotFoundError(
            f'No {split} report files found in {output_dir}'
        )

    summary = pd.DataFrame(rows)
    summary_path = output_dir / f'{split}_stage2_metrics_summary.csv'
    summary.to_csv(summary_path, index=False)
    summaries_by_split[split] = summary

    print(f'{split.upper()} summary saved to: {summary_path}')
    display(summary)

VALIDATION summary saved to: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/Stage 2 - Reranking/validation/validation_stage2_metrics_summary.csv


,model,stage,split,requests,known_click_recall@5,observed_click_ndcg@5,known_click_recall@100,observed_click_ndcg@100,known_click_recall@200,observed_click_ndcg@200
0,lstm_validation_stage1_base_top5_report,lstm_validation_stage1_base_top5_report,validation,29110,0.182542,0.109382,None,None,None,None
1,lstm_validation_stage2_diversity_only_top5_report,lstm_validation_stage2_diversity_only_top5_report,validation,29110,0.163448,0.099278,None,None,None,None
2,xgboost_validation_stage1_base_top5_report,xgboost_validation_stage1_base_top5_report,validation,29110,0.096848,0.059720,None,None,None,None
3,xgboost_validation_stage2_diversity_only_top5_...,xgboost_validation_stage2_diversity_only_top5_...,validation,29110,0.089449,0.056152,None,None,None,None
4,gemini_validation_stage1_base_top5_report,gemini_validation_stage1_base_top5_report,validation,29110,0.000618,0.000373,None,None,None,None
5,gemini_validation_stage2_diversity_only_top5_r...,gemini_validation_stage2_diversity_only_top5_r...,validation,29110,0.000721,0.000405,None,None,None,None
6,rrf_lstm75_xgboost25_validation_stage1_base_to...,rrf_lstm75_xgboost25_validation_stage1_base_to...,validation,29110,0.226086,0.143691,None,None,None,None
7,rrf_lstm75_xgboost25_validation_stage2_diversi...,rrf_lstm75_xgboost25_validation_stage2_diversi...,validation,29110,0.201308,0.130493,None,None,None,None


TEST summary saved to: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Evaluation/Stage 2 - Reranking/test/test_stage2_metrics_summary.csv


,model,stage,split,requests,known_click_recall@5,observed_click_ndcg@5,known_click_recall@100,observed_click_ndcg@100,known_click_recall@200,observed_click_ndcg@200
0,rrf_lstm75_xgboost25_test_stage1_base_top5_report,rrf_lstm75_xgboost25_test_stage1_base_top5_report,test,29142,0.230529,0.1467,None,None,None,None
1,rrf_lstm75_xgboost25_test_stage2_diversity_onl...,rrf_lstm75_xgboost25_test_stage2_diversity_onl...,test,29142,0.206940,0.1346,None,None,None,None
